In [1]:
!pip install spacy scikit-learn pandas numpy -q

In [3]:
import re
import random
import pandas as pd
import numpy as np
import spacy
from spacy.matcher import PhraseMatcher
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity

random.seed(42)

SKILL_TAXONOMY = {
    "python": ["python", "python3", "py"],
    "machine learning": ["machine learning", "ml", "machine-learning"],
    "deep learning": ["deep learning", "dl", "neural networks"],
    "nlp": ["nlp", "natural language processing", "text mining"],
    "sql": ["sql", "mysql", "postgresql", "structured query language"],
    "data analysis": ["data analysis", "data analytics", "data wrangling"],
    "tensorflow": ["tensorflow", "tf"],
    "pytorch": ["pytorch", "torch"],
    "scikit-learn": ["scikit-learn", "sklearn", "scikit learn"],
    "pandas": ["pandas"],
    "tableau": ["tableau"],
    "power bi": ["power bi", "powerbi"],
    "java": ["java", "core java"],
    "javascript": ["javascript", "js", "es6"],
    "react": ["react", "reactjs", "react.js"],
    "node.js": ["node.js", "nodejs", "node"],
    "html": ["html", "html5"],
    "css": ["css", "css3"],
    "rest api": ["rest api", "restful api", "rest apis"],
    "docker": ["docker", "containerization"],
    "aws": ["aws", "amazon web services"],
    "git": ["git", "github", "version control"],
    "communication": ["communication skills", "stakeholder communication"],
    "leadership": ["leadership", "team leadership", "people management"],
    "project management": ["project management", "agile", "scrum"],
}

In [4]:
ROLE_PROFILES = {
    "Data Scientist": {
        "core": ["python", "machine learning", "sql", "data analysis", "pandas", "scikit-learn"],
        "bonus": ["deep learning", "tensorflow", "nlp", "tableau", "aws"],
    },
    "Frontend Developer": {
        "core": ["javascript", "react", "html", "css", "git"],
        "bonus": ["node.js", "rest api", "docker", "communication"],
    },
    "Backend Developer": {
        "core": ["java", "sql", "rest api", "git", "docker"],
        "bonus": ["aws", "node.js", "project management", "leadership"],
    },
}

NAMES = ["Aarav","Ishita","Rohan","Priya","Karan","Sneha","Vikram","Ananya",
         "Dev","Meera","Arjun","Tanya","Nikhil","Pooja","Sanjay","Riya"]

def make_resume_text(name, skills, years_exp, role):
    skill_sentence = ", ".join(skills)
    return (f"{name} is a {role} with {years_exp} years of experience. "
            f"Core technical skills include {skill_sentence}. "
            f"Worked on multiple projects involving {random.choice(skills)} and "
            f"{random.choice(skills)}. Strong problem solver and team player. "
            f"Education: B.Tech in Computer Science.")

def generate_resumes(n=12):
    rows = []
    for i in range(n):
        role = random.choice(list(ROLE_PROFILES.keys()))
        profile = ROLE_PROFILES[role]
        n_core = random.randint(2, len(profile["core"]))
        n_bonus = random.randint(0, len(profile["bonus"]))
        skills = random.sample(profile["core"], n_core) + random.sample(profile["bonus"], n_bonus)
        years_exp = random.randint(0, 8)
        name = random.choice(NAMES) + f"_{i}"
        rows.append({"candidate_id": i, "name": name, "target_role": role,
                     "years_experience_actual": years_exp,
                     "resume_text": make_resume_text(name, skills, years_exp, role)})
    return pd.DataFrame(rows)

def make_job_description(role):
    profile = ROLE_PROFILES[role]
    required, preferred, min_years = profile["core"], profile["bonus"], 2
    jd = (f"We are hiring a {role}.\n"
          f"Required skills: {', '.join(required)}.\n"
          f"Preferred skills: {', '.join(preferred)}.\n"
          f"Minimum {min_years} years of experience required.")
    return jd, required, preferred, min_years

resumes_df = generate_resumes(12)
resumes_df.head()



,candidate_id,name,target_role,years_experience_actual,resume_text
0,0,Karan_0,Backend Developer,3,Karan_0 is a Backend Developer with 3 years of...
1,1,Vikram_1,Backend Developer,8,Vikram_1 is a Backend Developer with 8 years o...
2,2,Priya_2,Data Scientist,5,Priya_2 is a Data Scientist with 5 years of ex...
3,3,Rohan_3,Data Scientist,6,Rohan_3 is a Data Scientist with 6 years of ex...
4,4,Dev_4,Backend Developer,6,Dev_4 is a Backend Developer with 6 years of e...


In [5]:
def clean_text(text):
    text = text.lower()
    text = re.sub(r"[^a-z0-9\s\.\,\+\#]", " ", text)
    return re.sub(r"\s+", " ", text).strip()

nlp = spacy.blank("en")          
matcher = PhraseMatcher(nlp.vocab, attr="LOWER")
for canon, aliases in SKILL_TAXONOMY.items():
    matcher.add(canon, [nlp.make_doc(a) for a in aliases])

def extract_skills(text):
    doc = nlp(clean_text(text))
    return {nlp.vocab.strings[mid] for mid, s, e in matcher(doc)}

def extract_years_experience(text):
    m = re.search(r"(\d+)\s*\+?\s*years?", text.lower())
    return int(m.group(1)) if m else 0

In [6]:
REQUIRED_WEIGHT, PREFERRED_WEIGHT = 2.0, 1.0

def score_candidate(resume_text, required_skills, preferred_skills, min_years, jd_text):
    resume_skills = extract_skills(resume_text)
    years = extract_years_experience(resume_text)

    matched_required = resume_skills & set(required_skills)
    matched_preferred = resume_skills & set(preferred_skills)
    missing_required = set(required_skills) - resume_skills
    missing_preferred = set(preferred_skills) - resume_skills

    total_w = REQUIRED_WEIGHT*len(required_skills) + PREFERRED_WEIGHT*len(preferred_skills)
    earned_w = REQUIRED_WEIGHT*len(matched_required) + PREFERRED_WEIGHT*len(matched_preferred)
    skill_score = earned_w / total_w if total_w else 0

    vec = TfidfVectorizer(stop_words="english")
    tfidf = vec.fit_transform([clean_text(resume_text), clean_text(jd_text)])
    semantic_score = cosine_similarity(tfidf[0], tfidf[1])[0][0]

    exp_score = min(years / min_years, 1.0) if min_years else 1.0
    final_score = 0.5*skill_score + 0.3*semantic_score + 0.2*exp_score

    return {"skill_score": round(skill_score,3), "semantic_score": round(semantic_score,3),
            "experience_score": round(exp_score,3), "final_score": round(final_score,3),
            "matched_required": sorted(matched_required), "matched_preferred": sorted(matched_preferred),
            "missing_required": sorted(missing_required), "missing_preferred": sorted(missing_preferred),
            "years_detected": years}

def explain(row):
    s = (f"Final Score: {row['final_score']*100:.1f}% | Skill Match: {row['skill_score']*100:.0f}% "
         f"({len(row['matched_required'])} required, {len(row['matched_preferred'])} preferred) | "
         f"Semantic Fit: {row['semantic_score']*100:.0f}% | Experience: {row['years_detected']}y")
    if row["missing_required"]:
        s += f" | MISSING (required): {', '.join(row['missing_required'])}"
    return s

In [7]:
target_role = "Data Scientist"
jd_text, required, preferred, min_years = make_job_description(target_role)
print(jd_text, "\n")

results = []
for _, r in resumes_df.iterrows():
    res = score_candidate(r["resume_text"], required, preferred, min_years, jd_text)
    res.update({"candidate_id": r["candidate_id"], "name": r["name"], "actual_role": r["target_role"]})
    results.append(res)

ranked = pd.DataFrame(results).sort_values("final_score", ascending=False).reset_index(drop=True)
ranked.insert(0, "rank", ranked.index + 1)

print(ranked[["rank","name","actual_role","final_score","skill_score","semantic_score","experience_score"]])
print("\n--- Explanations (top 5) ---")
for _, row in ranked.head(5).iterrows():
    print(f"#{row['rank']} {row['name']} ({row['actual_role']}): {explain(row)}")

We are hiring a Data Scientist.
Required skills: python, machine learning, sql, data analysis, pandas, scikit-learn.
Preferred skills: deep learning, tensorflow, nlp, tableau, aws.
Minimum 2 years of experience required. 

    rank       name        actual_role  final_score  skill_score  \
0      1   Rohan_11     Data Scientist        0.821        0.941   
1      2    Priya_2     Data Scientist        0.758        0.824   
2      3  Vikram_10     Data Scientist        0.605        0.588   
3      4    Rohan_3     Data Scientist        0.604        0.588   
4      5   Sanjay_5     Data Scientist        0.598        0.588   
5      6      Dev_4  Backend Developer        0.320        0.176   
6      7    Priya_9  Backend Developer        0.314        0.176   
7      8   Vikram_1  Backend Developer        0.287        0.118   
8      9    Rohan_6  Backend Developer        0.287        0.118   
9     10    Karan_7  Backend Developer        0.282        0.118   
10    11    Karan_0  Backend 